# 04. Um agente com LangGraph

Vamos automatizar o ciclo que fizemos manualmente: chamar o modelo, executar as ferramentas solicitadas e devolver os resultados. No LangGraph, o estado guarda os dados, os nós executam funções e as arestas definem a próxima etapa.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

## Separe a conversa dos dados do tutor

`messages` guarda as mensagens trocadas. Já `current_topic`, `current_question_id`, `score` e `attempts` registram o tema, a pergunta atual, os acertos e as tentativas. Declarar esses campos não os atualiza automaticamente: precisamos escrever as regras de atualização.

In [ ]:
from typing import NotRequired
from langgraph.graph import MessagesState

class TutorState(MessagesState):
    current_topic: NotRequired[str | None]
    current_question_id: NotRequired[str | None]
    score: NotRequired[int]
    attempts: NotRequired[int]

## Monte o ciclo básico

O nó `agent` chama o modelo. Se ele pedir uma ferramenta, `tools_condition` encaminha a execução para `tools`. O resultado volta ao `agent`, que pode responder ou pedir outra ferramenta. Sem um novo pedido, o fluxo termina.

Esse primeiro grafo ainda não atualiza a pontuação nem salva o histórico entre chamadas. Execute a construção do grafo e depois a pergunta de exemplo.

In [ ]:
from langchain.messages import SystemMessage
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

from science_tutor.models import get_text_model
from science_tutor.tools import TOOLS

mini_model = get_text_model(temperature=0).bind_tools(TOOLS)

def agent(state: TutorState):
    system = SystemMessage(content="Você é um tutor de ciências. Use as ferramentas para consultar a base de conhecimento local e responda em português.")
    answer = mini_model.invoke([system, *state["messages"]])
    return {"messages": [answer]}

builder = StateGraph(TutorState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode(TOOLS))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition)
builder.add_edge("tools", "agent")

mini_graph = builder.compile()
print(mini_graph.get_graph().draw_mermaid())


In [ ]:
from langchain.messages import HumanMessage

mini_result = mini_graph.invoke({
    "messages": [HumanMessage(content="Quais temas de ciências posso estudar? Responda em português.")]
})
print(mini_result["messages"][-1].content)


### O que falta para o tutor completo?

Além de executar ferramentas, precisamos:

- Registrar o tema e a pergunta escolhidos.
- Atualizar acertos e tentativas após a correção.
- Recuperar a conversa nas próximas chamadas.

A versão completa inclui uma etapa de atualização do estado e um checkpointer, que guarda o estado de cada conversa durante a execução do programa.

## Use o grafo completo

Abra `src/science_tutor/graph.py` para acompanhar a implementação. O ciclo passa por `agent`, `tools`, `sync` e volta ao `agent`. A função `sync_state`, usada no nó `sync`, atualiza os dados do tutor a partir dos resultados das ferramentas.

Execute as células abaixo. O mesmo `thread_id` identifica a conversa nas duas chamadas. Depois da segunda, confira a pergunta atual e a pontuação no estado retornado.

In [ ]:
from science_tutor.graph import build_graph

graph = build_graph()
print(graph.get_graph().draw_mermaid())


In [ ]:
config = {"configurable": {"thread_id": "demo-01"}, "recursion_limit": 12}

result = graph.invoke(
    {"messages": [HumanMessage(content="Liste os temas de ciências que posso estudar.")]},
    config=config,
)
print(result["messages"][-1].content)


In [ ]:
result = graph.invoke(
    {"messages": [HumanMessage(content="Faça uma pergunta sobre células.")]},
    config=config,
)
print(result["messages"][-1].content)
print({
    key: result.get(key)
    for key in ["current_topic", "current_question_id", "score", "attempts"]
})


## Desafio 5: acompanhe as atualizações

Localize no código onde `attempts` e `score` mudam. Explique por que novas mensagens precisam ser incorporadas ao histórico, enquanto uma atualização de `score` substitui o valor anterior. Dica: confira o comportamento herdado de `MessagesState` e os valores retornados por `sync_state`.

## Desafio 6: conecte o agente à API

Abra `starter/api.py` e implemente `POST /chat`. Use `src/science_tutor/api.py` como referência para os campos da requisição e da resposta.

Envie duas requisições com o mesmo `thread_id` e uma terceira com outro identificador. Confira se as duas primeiras compartilham o histórico e se a terceira inicia uma conversa separada. Depois reinicie o servidor e observe o que acontece com o estado salvo em memória.